## ___Updating the mycorrhizal states___
--------------------

In [1]:
!python --version

Python 3.13.7


In [1]:
from urllib.request import Request, urlopen

import numpy as np
import pandas as pd
from bs4 import BeautifulSoup

In [2]:
# https://datadryad.org/dataset/doi:10.5061/dryad.n8bm9
maherali_crude = pd.read_excel(r"../../data/chapter2/Maherali.etal.AmNat.Data.xlsx", sheet_name="Original states data", skiprows=range(2))
maherali_crude.rename(mapper={old: old.replace('(', '').replace(')', '').lower().replace(' ', '_') for old in maherali_crude.columns}, axis=1, inplace=True) # column names have parentheses and spaces
# taxonomy columns in Maherali et. al. dataset has trailing spaces :(
maherali_crude.loc[:, "original_name_genus_species"] = maherali_crude.original_name_genus_species.str.strip()
maherali_crude.loc[:, "raw_state_record_from_publication"] = maherali_crude.raw_state_record_from_publication.str.strip()

final_maherali = pd.read_excel(r"../../data/chapter2/Maherali.etal.AmNat.Data.xlsx", sheet_name="Final list matched with phylo", skiprows=range(2))
final_maherali.rename(mapper={old: old.lower().replace(' ', '_') for old in final_maherali.columns}, axis=1, inplace=True)
final_maherali.genus_species = final_maherali.genus_species.str.strip().str.replace('_', ' ') # the sheet "Final list matched with phylo" has genus and specific epithets concatenated by under scores!

# in TRY, mycorrhiza type is trait id 7
try_myco = pd.read_csv(r"../../data/chapter2/TRY/mycorrhizal_states.txt", delimiter='\t', low_memory=False, encoding="latin1", usecols=["Dataset", "SpeciesName", "AccSpeciesName", "OrigValueStr",
                                "TraitID"]).dropna(subset=["AccSpeciesName", "OrigValueStr", "TraitID"])
# unify the mycorrhizal state info
# 'ECTO', 'NM/AM', 'EC', 'EC/AM', 'AM', 'Ecto', 'Non',        'vesicular-arbuscular mycorrhiza', 'ectomycorrhiza', 'no', '0', 'Ph.th.end.', 'VAM', 'Ectomycorrhiza', 'E.ch.ect.', 'arbuscular',
# 'ec?', 'VA', 'ecto', 'Absent', 'non-ectomycorrhizal', 'ectomycorrhizal', 'Yes', 'No', 'EM', 'AMNM', 'NM', 'AM + EM', 'ERM', 'Ericoid', 'ECM'

MYCORRHIZAL_STATES_REPLACEMENTS = {
    "ECTO": "EM",
    "Ecto": "EM",
    "EC": "EM",
    "ectomycorrhiza": "EM",
    "Ectomycorrhiza": "EM",
    "ecto": "EM",
    "ectomycorrhizal": "EM",
    "ECM": "EM",
    "vesicular-arbuscular mycorrhiza" : "AM",
    "VAM": "AM",
    "VA": "AM",
    "Non": "NM",
    "AMNM": "NM/AM",
    "Ericoid": "ER",
    "ERM": "ER",
    "AM + EM": "AM/EM",
    "EC/AM": "AM/EM",
    "Orchid": "OM",
    "OrM": "OM"
}

try_myco.loc[:, "OrigValueStr"] = try_myco.OrigValueStr.replace(MYCORRHIZAL_STATES_REPLACEMENTS)
try_myco = try_myco.query("OrigValueStr.isin(@MYCORRHIZAL_STATES_REPLACEMENTS.values())")

mycodb_v4 = pd.read_csv(r"../../data/chapter2/MycoDB_version4.csv", usecols=["PlantSpecies2018", "FUNGROUP", "MYCORRHIZAETYPE", "AM_single_genus", "EM_single_genus", 
                        "STERILIZED", "NONMYCOCONTROL", "NONMYCOCONTROL2"]).dropna(subset="PlantSpecies2018").drop_duplicates()
mycodb_v4.loc[:, "PlantSpecies2018"] = mycodb_v4.PlantSpecies2018.str.capitalize().str.replace('_', ' ')

# scrape the online only MycoDB metadata and serialize it to the disk
# req = Request(url=r"https://www.nature.com/articles/sdata201628/tables/2", headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:142.0) Gecko/20100101 Firefox/142.0"})
# with urlopen(req) as r:
#     soup = BeautifulSoup(r.read())
# 
# table = soup.find(name="table", attrs={"class": "data last-table"}) # locate the metadata table
# [th.text.strip() for th in table.find_all(name="th")] # column names
# mycodb_descriptions = [[td.text for td in tr.find_all(name="td")] for tr in table.find_all(name="tr")[1:]] # parse the rows
# 
# # create a dataframe using the parsed rows and column names and serialize it to the disk
# pd.DataFrame({ 
#     "Variable": [row[0] for row in mycodb_descriptions],
#     "Description": [row[1] for row in mycodb_descriptions],
#     "Variable Type (range)": [row[2] for row in mycodb_descriptions],
#     "Levels (#studies/level)": [row[3] for row in mycodb_descriptions],
# }).to_csv(r"../data/chapter2/MycoDB_version4_metadata.csv", index=False)

mycodb_v4_meta = pd.read_csv(r"../../data/chapter2/MycoDB_version4_metadata.csv")

subset_categorical = pd.read_csv(r"../../data/chapter2/FREDv3subset/FRED_subset_categorical.csv")

In [31]:
subset_categorical

,binominal,F01286,F01287,F01289,F01290,F00043,F00645,F00004
0,Populus trichocarpa,Populus,trichocarpa,Salicaceae,Malpighiales,C3,NaN,"Hajek P, Hertel D, Leuschner C. 2014. Root ord..."
1,Populus tremula,Populus,tremula,Salicaceae,Malpighiales,C3,NaN,"Hajek P, Hertel D, Leuschner C. 2014. Root ord..."
2,Altingia obovata,Altingia,obovata,Altingiaceae,Saxifragales,C3,NaN,"Xu Y, Gu JC, Dong XY, Liu Y, Wang ZQ. 2011. Fi..."
3,Cryptocarya chinensis,Cryptocarya,chinensis,Lauraceae,Laurales,C3,NaN,"Xu Y, Gu JC, Dong XY, Liu Y, Wang ZQ. 2011. Fi..."
4,Elaeocarpus sylvestris,Elaeocarpus,sylvestris,Elaeocarpaceae,Oxalidales,C3,NaN,"Xu Y, Gu JC, Dong XY, Liu Y, Wang ZQ. 2011. Fi..."
...,...,...,...,...,...,...,...,...
236,Populus deltoides,Populus,deltoides,Salicaceae,Malpighiales,C3,NaN,Valverde et al (unpublished)
237,Prunus sargentii,Prunus,sargentii,Rosaceae,Rosales,C3,NaN,Valverde et al (unpublished)
238,Styphnolobium japonicum,Styphnolobium,japonicum,Fabaceae,Fabales,C3,NaN,Valverde et al (unpublished)
239,Syringa reticulata,Syringa,reticulata,Oleaceae,Lamiales,C3,NaN,Valverde et al (unpublished)


In [20]:
 subset_categorical.F00645.unique()

array([nan, 'AM', 'EM', 'AM + EM', 'NM', 'ErM'], dtype=object)

In [23]:
try_myco.OrigValueStr.unique()

array(['NM/AM', 'EM', 'AM/EM', 'ER', 'AM', 'NM', 'OM'], dtype=object)

In [22]:
pd.merge(left=subset_categorical, left_on="binominal", right=try_myco, right_on="AccSpeciesName", how="inner").drop_duplicates(subset=["binominal", "OrigValueStr"]).OrigValueStr.unique()

array(['EM', 'NM', 'AM', 'NM/AM', 'AM/EM', 'ER'], dtype=object)

In [24]:
groot = pd.read_csv(r"../../data/chapter2/GRooTFullVersion.csv", encoding="latin1", low_memory=False)
groot.insert(loc=0, column="binominal", value=groot.genus.str.capitalize().str.strip() + ' ' + groot.species.str.lower().str.strip())

In [30]:
groot.mycorrhizalAssociationType.isna().mean(), groot.mycorrhizalAssociationTypeFungalRoot.isna().mean()

(np.float64(0.8380171945859817), np.float64(0.019497119644201644))

In [34]:
groot_myco = groot.query("not mycorrhizalAssociationTypeFungalRoot.isna()").loc[:, ["binominal", "mycorrhizalAssociationTypeFungalRoot"]].drop_duplicates()
groot_myco

,binominal,mycorrhizalAssociationTypeFungalRoot
0,NaN,EcM
5,Agropyron cristatum,AM
6,Artemisia tridentata,AM
7,Chrysothamnus viscidiflorus,AM
8,Elymus elymoides,AM
...,...,...
114201,Valeriana excelsa,AM
114202,Veronica aphylla,NM-AM
114204,Vicia onobrychioides,AM
114205,Vicia oroboides,AM


In [36]:
pd.merge(left=subset_categorical.query("F00645.isna()"), left_on="binominal", right=groot_myco, right_on="binominal")

,binominal,F01286,F01287,F01289,F01290,F00043,F00645,F00004,mycorrhizalAssociationTypeFungalRoot
0,Populus trichocarpa,Populus,trichocarpa,Salicaceae,Malpighiales,C3,NaN,"Hajek P, Hertel D, Leuschner C. 2014. Root ord...",EcM-AM
1,Populus tremula,Populus,tremula,Salicaceae,Malpighiales,C3,NaN,"Hajek P, Hertel D, Leuschner C. 2014. Root ord...",EcM-AM
2,Altingia obovata,Altingia,obovata,Altingiaceae,Saxifragales,C3,NaN,"Xu Y, Gu JC, Dong XY, Liu Y, Wang ZQ. 2011. Fi...",AM
3,Cryptocarya chinensis,Cryptocarya,chinensis,Lauraceae,Laurales,C3,NaN,"Xu Y, Gu JC, Dong XY, Liu Y, Wang ZQ. 2011. Fi...",AM
4,Elaeocarpus sylvestris,Elaeocarpus,sylvestris,Elaeocarpaceae,Oxalidales,C3,NaN,"Xu Y, Gu JC, Dong XY, Liu Y, Wang ZQ. 2011. Fi...",AM
5,Endospermum chinense,Endospermum,chinense,Euphorbiaceae,Malpighiales,C3,NaN,"Xu Y, Gu JC, Dong XY, Liu Y, Wang ZQ. 2011. Fi...",AM
6,Ardisia quinquegona,Ardisia,quinquegona,Primulaceae,Ericales,C3,NaN,"Wang J-J, Tharayil N, Chow AT, Suseela V, Zeng...",AM
7,Cystopteris sudetica,Cystopteris,sudetica,Cystopteridaceae,Polypodiales,C3,NaN,"Dong X, Wang H, Gu J, Wang Y, Wang Z. 2014. Ro...",AM
8,Equisetum pratense,Equisetum,pratense,Equisetaceae,Equisetales,C3,NaN,"Dong X, Wang H, Gu J, Wang Y, Wang Z. 2014. Ro...",NM-AM
9,Matteuccia struthiopteris,Matteuccia,struthiopteris,Onocleaceae,Polypodiales,C3,NaN,"Dong X, Wang H, Gu J, Wang Y, Wang Z. 2014. Ro...",AM


In [7]:
fungalroot = pd.read_csv(r"../../data/chapter2/FungalRoot/occurrence.txt", sep='\t', low_memory=False)

In [12]:
fungalroot.columns.str.contains("myco").sum()

np.int64(0)

In [15]:
fungalroot.columns[fungalroot.columns.str.contains("scient")]

Index(['scientificNameID', 'scientificName'], dtype='object')

In [18]:
fungalroot.scientificName.drop_duplicates()

0                                       Juncus inflexus L.
1                       Hygrophila balsamica (L.fil.) Raf.
2                                          Cyperus iria L.
3                                 Cyperus microiria Steud.
4                   Arachis duranensis Krapov. & W.C.Greg.
                               ...                        
36487    Allochrusa paniculata (Regel & Herder) Ovcz. &...
36494    Acer platanoides subsp. turkestanicum (Pax) P....
36508           Acanthophyllum korolkowii Regel & Schmalh.
36509      Abies sachalinensis var. mayriana Miyabe & Kudô
36513                             Abies koreana E.H.Wilson
Name: scientificName, Length: 14528, dtype: object

In [30]:
fungalroot_genus_level = pd.read_excel(r"../../data/chapter2/FungalRoot/nph16569-sup-0002-tabless1-s4.xlsx", sheet_name="Table S2", skiprows=range(2), header=0)

In [31]:
fungalroot_genus_level

,Genus,Mycorrhizal type
0,Aa,OM
1,Aaronsohnia,AM
2,Abarema,AM
3,Abatia,AM
4,Abdra,NM
...,...,...
14536,Yulania,AM
14537,Yungasocereus,AM
14538,Yungastocactus,AM
14539,Yushania,AM


In [36]:
subset_categorical.query("F00645.isna()").drop_duplicates(subset="F01286")

,binominal,F01286,F01287,F01289,F01290,F00043,F00645,F00004
0,Populus trichocarpa,Populus,trichocarpa,Salicaceae,Malpighiales,C3,NaN,"Hajek P, Hertel D, Leuschner C. 2014. Root ord..."
2,Altingia obovata,Altingia,obovata,Altingiaceae,Saxifragales,C3,NaN,"Xu Y, Gu JC, Dong XY, Liu Y, Wang ZQ. 2011. Fi..."
3,Cryptocarya chinensis,Cryptocarya,chinensis,Lauraceae,Laurales,C3,NaN,"Xu Y, Gu JC, Dong XY, Liu Y, Wang ZQ. 2011. Fi..."
4,Elaeocarpus sylvestris,Elaeocarpus,sylvestris,Elaeocarpaceae,Oxalidales,C3,NaN,"Xu Y, Gu JC, Dong XY, Liu Y, Wang ZQ. 2011. Fi..."
5,Endospermum chinense,Endospermum,chinense,Euphorbiaceae,Malpighiales,C3,NaN,"Xu Y, Gu JC, Dong XY, Liu Y, Wang ZQ. 2011. Fi..."
45,Ardisia quinquegona,Ardisia,quinquegona,Primulaceae,Ericales,C3,NaN,"Wang J-J, Tharayil N, Chow AT, Suseela V, Zeng..."
144,Cystopteris sudetica,Cystopteris,sudetica,Cystopteridaceae,Polypodiales,C3,NaN,"Dong X, Wang H, Gu J, Wang Y, Wang Z. 2014. Ro..."
146,Equisetum pratense,Equisetum,pratense,Equisetaceae,Equisetales,C3,NaN,"Dong X, Wang H, Gu J, Wang Y, Wang Z. 2014. Ro..."
147,Matteuccia struthiopteris,Matteuccia,struthiopteris,Onocleaceae,Polypodiales,C3,NaN,"Dong X, Wang H, Gu J, Wang Y, Wang Z. 2014. Ro..."
148,Onoclea sensibilis,Onoclea,sensibilis,Onocleaceae,Polypodiales,C3,NaN,"Dong X, Wang H, Gu J, Wang Y, Wang Z. 2014. Ro..."


In [37]:
pd.merge(left=subset_categorical.query("F00645.isna()").drop_duplicates(subset="F01286"), left_on="F01286", right=fungalroot_genus_level, right_on="Genus", how="left")

,binominal,F01286,F01287,F01289,F01290,F00043,F00645,F00004,Genus,Mycorrhizal type
0,Populus trichocarpa,Populus,trichocarpa,Salicaceae,Malpighiales,C3,NaN,"Hajek P, Hertel D, Leuschner C. 2014. Root ord...",Populus,EcM-AM
1,Altingia obovata,Altingia,obovata,Altingiaceae,Saxifragales,C3,NaN,"Xu Y, Gu JC, Dong XY, Liu Y, Wang ZQ. 2011. Fi...",Altingia,AM
2,Cryptocarya chinensis,Cryptocarya,chinensis,Lauraceae,Laurales,C3,NaN,"Xu Y, Gu JC, Dong XY, Liu Y, Wang ZQ. 2011. Fi...",Cryptocarya,AM
3,Elaeocarpus sylvestris,Elaeocarpus,sylvestris,Elaeocarpaceae,Oxalidales,C3,NaN,"Xu Y, Gu JC, Dong XY, Liu Y, Wang ZQ. 2011. Fi...",Elaeocarpus,AM
4,Endospermum chinense,Endospermum,chinense,Euphorbiaceae,Malpighiales,C3,NaN,"Xu Y, Gu JC, Dong XY, Liu Y, Wang ZQ. 2011. Fi...",Endospermum,AM
5,Ardisia quinquegona,Ardisia,quinquegona,Primulaceae,Ericales,C3,NaN,"Wang J-J, Tharayil N, Chow AT, Suseela V, Zeng...",Ardisia,AM
6,Cystopteris sudetica,Cystopteris,sudetica,Cystopteridaceae,Polypodiales,C3,NaN,"Dong X, Wang H, Gu J, Wang Y, Wang Z. 2014. Ro...",Cystopteris,AM
7,Equisetum pratense,Equisetum,pratense,Equisetaceae,Equisetales,C3,NaN,"Dong X, Wang H, Gu J, Wang Y, Wang Z. 2014. Ro...",Equisetum,NM-AM
8,Matteuccia struthiopteris,Matteuccia,struthiopteris,Onocleaceae,Polypodiales,C3,NaN,"Dong X, Wang H, Gu J, Wang Y, Wang Z. 2014. Ro...",Matteuccia,AM
9,Onoclea sensibilis,Onoclea,sensibilis,Onocleaceae,Polypodiales,C3,NaN,"Dong X, Wang H, Gu J, Wang Y, Wang Z. 2014. Ro...",Onoclea,AM
